# ExomSystem — Pruebas DAO
Este notebook prueba todas las operaciones CRUD del proyecto CivicTech.

> **Requisito:** tener el `docker-compose up` corriendo antes de ejecutar.

## 0. Imports

In [2]:
import sys
import os

# Asegura que Python encuentra dao.py y db_models/ desde /app (raíz del proyecto)
sys.path.insert(0, '/app')

from datetime import datetime
from dao import ConexionDB, UsuarioDAO, ReporteDAO, EvidenciaDAO
from db_models import Usuario, Reporte, Evidencia

print('Imports OK ✅')
print('sys.path[0]:', sys.path[0])

Imports OK ✅
sys.path[0]: /app


## 1. Verificar conexión a la base de datos

In [3]:
# Las variables de entorno las inyecta docker-compose automáticamente
print('DB_HOST :', os.getenv('DB_HOST'))
print('DB_NAME :', os.getenv('DB_NAME'))
print('DB_USER :', os.getenv('DB_USER'))

with ConexionDB() as conn:
    print('\n✅ Conexión exitosa a PostgreSQL + PostGIS')

DB_HOST : db
DB_NAME : civictech_db
DB_USER : postgres

✅ Conexión exitosa a PostgreSQL + PostGIS


## 2. UsuarioDAO — CRUD

In [4]:
# ── Crear usuario ──────────────────────────────────────────
with ConexionDB() as conn:
    dao = UsuarioDAO(conn)
    usuario = dao.crear(Usuario(
        nombre_completo='Ana Pérez',
        dni='30123456',
        email='ana@civictech.com',
        contrasena='password123'
    ))
    print('Creado :', usuario)

UndefinedColumn: column "dni" of relation "usuario" does not exist
LINE 2: ...          INSERT INTO "usuario" (nombre_apellido, dni, email...
                                                             ^


In [5]:
# ── Obtener por ID ─────────────────────────────────────────
with ConexionDB() as conn:
    dao = UsuarioDAO(conn)
    u = dao.obtener_por_id(usuario.id_usuario)
    print('Por ID  :', u)

NameError: name 'usuario' is not defined

In [5]:
# ── Obtener por email ──────────────────────────────────────
with ConexionDB() as conn:
    dao = UsuarioDAO(conn)
    u = dao.obtener_por_email('ana@civictech.com')
    print('Por email:', u)

UndefinedColumn: column "dni" does not exist
LINE 2:             SELECT id_usuario, nombre_apellido, dni, email, ...
                                                        ^


In [6]:
# ── Actualizar reputación ──────────────────────────────────
with ConexionDB() as conn:
    dao = UsuarioDAO(conn)
    ok = dao.actualizar_reputacion(usuario.id_usuario, 95)
    print('Reputación actualizada:', ok)

NameError: name 'usuario' is not defined

## 3. ReporteDAO — CRUD

In [6]:
# ── Crear reporte ──────────────────────────────────────────
# ubicacion = (longitud, latitud) — coordenadas de Chilecito, La Rioja
with ConexionDB() as conn:
    dao = ReporteDAO(conn)
    reporte = dao.crear(Reporte(
        id_usuario=usuario.id_usuario,
        patente_vehiculo='AB123CD',
        fecha_hora_dispositivo=datetime.now(),
        ubicacion=(-67.4965, -29.1611),
        descripcion='Prueba desde Jupyter'
    ))
    print('Creado :', reporte)
    print('Fecha server:', reporte.fecha_hora_server)

NameError: name 'usuario' is not defined

In [7]:
# ── Obtener por ID ─────────────────────────────────────────
with ConexionDB() as conn:
    dao = ReporteDAO(conn)
    r = dao.obtener_por_id(reporte.id_reporte)
    print('Por ID:', r)
    print('  GPS (lon, lat):', r.ubicacion)
    print('  Estado:', r.estado)

NameError: name 'reporte' is not defined

In [8]:
# ── Obtener por patente ────────────────────────────────────
with ConexionDB() as conn:
    dao = ReporteDAO(conn)
    lista = dao.obtener_por_patente('AB123CD')
    print(f'Reportes para AB123CD: {len(lista)}')
    for r in lista:
        print(' -', r)

Reportes para AB123CD: 0


In [9]:
# ── Obtener por estado ─────────────────────────────────────
with ConexionDB() as conn:
    dao = ReporteDAO(conn)
    pendientes = dao.obtener_por_estado('Pendiente')
    print(f'Reportes pendientes: {len(pendientes)}')

Reportes pendientes: 1


In [10]:
# ── Consulta espacial PostGIS: radio de 1 km ───────────────
with ConexionDB() as conn:
    dao = ReporteDAO(conn)
    cercanos = dao.obtener_en_radio(
        lon=-67.4965, lat=-29.1611, metros=1000
    )
    print(f'Reportes en radio 1 km: {len(cercanos)}')
    for r in cercanos:
        # BUGFIX: r.ubicacion (no coordenadas_gps), r.estado (no estado_resolucion)
        print(f'  id={r.id_reporte}  GPS={r.ubicacion}  estado={r.estado}')

Reportes en radio 1 km: 3
  id=1  GPS=(-67.4965, -29.1611)  estado=Pendiente
  id=2  GPS=(-67.4892, -29.1664)  estado=Validada
  id=3  GPS=(-67.4915, -29.1548)  estado=Rechazada


In [11]:
# ── Actualizar estado ──────────────────────────────────────
# Estados válidos según CHECK constraint: 'Pendiente' | 'Validada' | 'Rechazada'
with ConexionDB() as conn:
    dao = ReporteDAO(conn)
    ok = dao.actualizar_estado(reporte.id_reporte, 'Validada')
    print('Estado actualizado:', ok)
    r = dao.obtener_por_id(reporte.id_reporte)
    # BUGFIX: r.estado (no r.estado_resolucion)
    print('Nuevo estado:', r.estado)

NameError: name 'reporte' is not defined

## 4. EvidenciaDAO — CRUD

In [ ]:
import hashlib

# Simular hash SHA-256 de una foto
hash_foto = hashlib.sha256(b'foto_de_prueba_civictech').hexdigest()
print('Hash SHA-256:', hash_foto)

with ConexionDB() as conn:
    dao = EvidenciaDAO(conn)
    # BUGFIX: id_infraccion (no id_reporte), hash_seguridad_sha (no hash_seguridad_sha256)
    evidencia = dao.crear(Evidencia(
        id_infraccion=reporte.id_reporte,
        url_archivo_s3='https://mi-bucket.s3.amazonaws.com/fotos/AB123CD_001.jpg',
        hash_seguridad_sha=hash_foto,
    ))
    print('Creada:', evidencia)

In [ ]:
# ── Obtener por reporte ────────────────────────────────────
with ConexionDB() as conn:
    dao = EvidenciaDAO(conn)
    lista = dao.obtener_por_reporte(reporte.id_reporte)
    print(f'Evidencias del reporte {reporte.id_reporte}: {len(lista)}')
    for e in lista:
        print(' -', e)

In [ ]:
# ── Buscar por hash (validación de integridad jurídica) ────
with ConexionDB() as conn:
    dao = EvidenciaDAO(conn)
    e = dao.obtener_por_hash(hash_foto)
    print('Evidencia encontrada por hash:', e)

## 5. Limpieza (opcional)
Ejecutar solo si querés borrar los datos de prueba.

In [ ]:
# Las evidencias y reportes se eliminan en cascada al borrar el usuario
with ConexionDB() as conn:
    dao = UsuarioDAO(conn)
    ok = dao.eliminar(usuario.id_usuario)
    print('Usuario eliminado (cascade):', ok)